In [86]:
from google.colab import drive

import os
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import warnings
warnings.filterwarnings("ignore")


In [32]:

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:

os.chdir("/content/drive/MyDrive/car-insurance-claim-prediction")


In [34]:
!pwd


/content/drive/MyDrive/car-insurance-claim-prediction


In [45]:

train = pd.read_csv("data/raw/train.csv")
test = pd.read_csv("data/raw/test.csv")
sample = pd.read_csv("data/raw/sample_submission.csv")

train.shape, test.shape, sample.shape


((58592, 44), (39063, 43), (39063, 2))

In [46]:
train.head()

,policy_id,policy_tenure,age_of_car,age_of_policyholder,area_cluster,population_density,make,segment,model,fuel_type,...,is_brake_assist,is_power_door_locks,is_central_locking,is_power_steering,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,is_claim
0,ID00001,0.515874,0.05,0.644231,C1,4990,1,A,M1,CNG,...,No,No,No,Yes,No,No,No,Yes,0,0
1,ID00002,0.672619,0.02,0.375000,C2,27003,1,A,M1,CNG,...,No,No,No,Yes,No,No,No,Yes,0,0
2,ID00003,0.841110,0.02,0.384615,C3,4076,1,A,M1,CNG,...,No,No,No,Yes,No,No,No,Yes,0,0
3,ID00004,0.900277,0.11,0.432692,C4,21622,1,C1,M2,Petrol,...,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,2,0
4,ID00005,0.596403,0.11,0.634615,C5,34738,2,A,M3,Petrol,...,No,Yes,Yes,Yes,No,Yes,Yes,Yes,2,0


In [36]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58592 entries, 0 to 58591
Data columns (total 43 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   policy_tenure                     58592 non-null  float64 
 1   age_of_car                        58592 non-null  float64 
 2   age_of_policyholder               58592 non-null  float64 
 3   area_cluster                      58592 non-null  string  
 4   population_density                58592 non-null  int64   
 5   make                              58592 non-null  int64   
 6   segment                           58592 non-null  category
 7   model                             58592 non-null  category
 8   fuel_type                         58592 non-null  category
 9   max_torque                        58592 non-null  string  
 10  max_power                         58592 non-null  string  
 11  engine_type                       58592 non-null  cate

In [47]:
c = train[['segment','model','fuel_type','engine_type','rear_brakes_type','transmission_type','steering_type'] ]

In [48]:
s = train[['policy_id','area_cluster','max_torque','max_power']]

In [49]:
b = train[['is_adjustable_steering','is_tpms' ,'is_parking_sensors','is_parking_camera','is_front_fog_lights' ,'is_rear_window_wiper', 'is_rear_window_washer','is_rear_window_defogger' ,'is_brake_assist' , 'is_power_door_locks', 'is_central_locking',  'is_power_steering' ,  'is_driver_seat_height_adjustable', 'is_day_night_rear_view_mirror', 'is_ecw' ,'is_speed_alert']]

In [72]:
train[s.columns] = train[s.columns].astype('string')
train[c.columns] = train[c.columns].astype('category')
train[b.columns] = train[b.columns].astype('bool')

In [75]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58592 entries, 0 to 58591
Data columns (total 43 columns):
 #   Column                            Non-Null Count  Dtype   
---  ------                            --------------  -----   
 0   policy_tenure                     58592 non-null  float64 
 1   age_of_car                        58592 non-null  float64 
 2   age_of_policyholder               58592 non-null  float64 
 3   area_cluster                      58592 non-null  string  
 4   population_density                58592 non-null  int64   
 5   make                              58592 non-null  int64   
 6   segment                           58592 non-null  category
 7   model                             58592 non-null  category
 8   fuel_type                         58592 non-null  category
 9   max_torque                        58592 non-null  string  
 10  max_power                         58592 non-null  string  
 11  engine_type                       58592 non-null  cate

In [74]:
train['is_esc'].astype('bool')

,is_esc
0,True
1,True
2,True
3,True
4,True
...,...
58587,True
58588,True
58589,True
58590,True


In [71]:
train['airbags'].astype('int64')

,airbags
0,2
1,2
2,2
3,2
4,2
...,...
58587,2
58588,2
58589,2
58590,2


In [40]:
train.head()

,policy_tenure,age_of_car,age_of_policyholder,area_cluster,population_density,make,segment,model,fuel_type,max_torque,...,is_brake_assist,is_power_door_locks,is_central_locking,is_power_steering,is_driver_seat_height_adjustable,is_day_night_rear_view_mirror,is_ecw,is_speed_alert,ncap_rating,is_claim
0,0.515874,0.05,0.644231,C1,4990,1,A,M1,CNG,60Nm@3500rpm,...,True,True,True,True,True,True,True,True,0,0
1,0.672619,0.02,0.375000,C2,27003,1,A,M1,CNG,60Nm@3500rpm,...,True,True,True,True,True,True,True,True,0,0
2,0.841110,0.02,0.384615,C3,4076,1,A,M1,CNG,60Nm@3500rpm,...,True,True,True,True,True,True,True,True,0,0
3,0.900277,0.11,0.432692,C4,21622,1,C1,M2,Petrol,113Nm@4400rpm,...,True,True,True,True,True,True,True,True,2,0
4,0.596403,0.11,0.634615,C5,34738,2,A,M3,Petrol,91Nm@4250rpm,...,True,True,True,True,True,True,True,True,2,0


In [41]:
train.columns

Index(['policy_tenure', 'age_of_car', 'age_of_policyholder', 'area_cluster',
       'population_density', 'make', 'segment', 'model', 'fuel_type',
       'max_torque', 'max_power', 'engine_type', 'airbags', 'is_esc',
       'is_adjustable_steering', 'is_tpms', 'is_parking_sensors',
       'is_parking_camera', 'rear_brakes_type', 'displacement', 'cylinder',
       'transmission_type', 'gear_box', 'steering_type', 'turning_radius',
       'length', 'width', 'height', 'gross_weight', 'is_front_fog_lights',
       'is_rear_window_wiper', 'is_rear_window_washer',
       'is_rear_window_defogger', 'is_brake_assist', 'is_power_door_locks',
       'is_central_locking', 'is_power_steering',
       'is_driver_seat_height_adjustable', 'is_day_night_rear_view_mirror',
       'is_ecw', 'is_speed_alert', 'ncap_rating', 'is_claim'],
      dtype='object')

In [50]:
train.columns = train.columns.str.strip()

In [51]:
train.drop('policy_id' ,axis = 1 ,errors = 'ignore' , inplace = True)

In [52]:
train.isna().sum().sum()


np.int64(0)

# ***ENCODING***

In [53]:
X = train.drop(columns = ['is_claim'])
y = train['is_claim']

In [55]:
s = s.drop(columns= 'policy_id')

In [57]:
n = ['length',
   'width',
   'height',
   'gross_weight',
    'ncap_rating',
   'is_claim',
   'gear_box',
      'displacement',
  'cylinder',
 'airbags',
  'population_density',
    'make'  ]

In [58]:
cat_cols = list(s.columns) + list(c.columns)


In [59]:
encoded = pd.get_dummies(
    train[cat_cols],
    drop_first=True
)


In [60]:
X_encoded = pd.concat(
    [
        train[n],          # numerical (unchanged)
        train[b.columns],  # boolean (unchanged)
        encoded             # one-hot encoded categoricals
    ],
    axis=1
)


In [61]:
X = X_encoded.drop(columns=['is_claim'])
y = X_encoded['is_claim']


# ***Train Test Split***

In [62]:


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [65]:
n = [col for col in n if col != 'is_claim']


In [66]:


scaler = StandardScaler()

X_train[n] = scaler.fit_transform(X_train[n])
X_test[n] = scaler.transform(X_test[n])


In [78]:
for col in b.columns:
  X_train[col] = X_train[col].map({'Yes':1 ,'No':0})
  X_test[col] = X_test[col].map({'Yes':1 , 'No':0})

# ***Logistic Regression***

In [79]:


log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_reg.fit(X_train, y_train)


LogisticRegression(max_iter=1000, random_state=42)

In [82]:

y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]


cm = confusion_matrix(y_test, y_pred)
cm

array([[10969,     0],
       [  750,     0]])

In [83]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.94      1.00      0.97     10969
           1       0.00      0.00      0.00       750

    accuracy                           0.94     11719
   macro avg       0.47      0.50      0.48     11719
weighted avg       0.88      0.94      0.91     11719



In [84]:
roc_auc_score(y_test, y_prob)


np.float64(0.5208728234114323)

In [87]:


rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)


RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [88]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "ROC-AUC": [
        roc_auc_score(y_test, log_reg.predict_proba(X_test)[:, 1]),
        roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
    ]
})

comparison



,Model,ROC-AUC
0,Logistic Regression,0.520873
1,Random Forest,0.525929


In [89]:
feature_importance = pd.Series(
    rf.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

feature_importance.head(15)


,0
population_density,0.208808
area_cluster_C21,0.048139
area_cluster_C18,0.043638
area_cluster_C3,0.036875
area_cluster_C7,0.032979
area_cluster_C4,0.031480
area_cluster_C19,0.028907
area_cluster_C8,0.028320
area_cluster_C5,0.026616
area_cluster_C14,0.026456


In [93]:
!mkdir -p models


In [94]:
import joblib

joblib.dump(rf, "models/random_forest_model.pkl")


['models/random_forest_model.pkl']

In [95]:


joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(X_train.columns.tolist(), "models/feature_columns.pkl")


['models/feature_columns.pkl']

In [97]:
!git status


On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/
	models/
	notebooks/

nothing added to commit but untracked files present (use "git add" to track)


In [98]:
!git add .
!git commit -m "Add preprocessing, baseline models, and saved artifacts"
!git push


Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@196fb0df17b2.(none)')
error: src refspec refs/heads/main does not match any
error: failed to push some refs to 'https://github.com/Ajey-Raj-Jha/car-insurance-claim-prediction.git'
